# Fundamentos de Big Data en la Nube (2025)

**Objetivo:** Entender los principios esenciales de arquitecturas Big Data modernas en la nube: almacenamiento distribuido, cómputo escalable y patrones arquitectónicos clave.

---
## 1. ¿Por qué Big Data sigue evolucionando en 2025?

Las demandas de volumen (PB+), variedad (estructurado, semiestructurado, no estructurado, vectorial) y velocidad (ms–segundos) impulsan nuevas generaciones de frameworks y servicios cloud.

| Dimensión | 2015 | 2020 | 2025 | Cambio Clave |
|-----------|------|------|------|--------------|
| Volumen | TB | 100s TB | PB+ | Optimización de formatos columnares y compresión adaptativa |
| Velocidad | Batch diario | Micro-batch | Near real-time + event streaming | Unificación batch/stream |
| Variedad | Principalmente estructurado | JSON + logs | Texto, imagen, audio, vectores | Modelos multimodales |
| Complejidad Operativa | Alta (DevOps manual) | Media | Menor (Serverless / Auto-scaling) | Automatización cloud |
| Observabilidad | Métricas básicas | Dashboards | Métricas + trazas + lineage integrado | Data Observability Platforms |

---
## 2. Capas de una Arquitectura Big Data en la Nube

```
┌───────────────────────────────┐
│        Ingesta (Edge)         │  ← IoT, APIs, Logs, Batches
└───────────────┬──────────────┘
                ▼
┌───────────────────────────────┐
│  Streaming / Event Bus        │  ← Kafka / Event Hub / PubSub
└───────────────┬──────────────┘
                ▼
┌───────────────────────────────────────────────┐
│         Data Lake / Lakehouse                 │
│ Bronze (Raw) / Silver (Curado) / Gold (BI/ML) │
│ Formatos: Parquet, Iceberg, Delta, Hudi       │
└───────────────┬──────────────────────────────┘
                ▼
┌───────────────────────────────┐    ┌────────────────────────┐
│  Batch & Stream Processing    │    │  Query Serving / BI     │
│ Spark / Flink / Beam / dbt    │    │ Warehouse / Materialize │
└───────────────┬──────────────┘    └────────┬───────────────┘
                ▼                               ▼
┌───────────────────────────────┐    ┌────────────────────────┐
│   ML / Feature Engineering    │    │  Data Products / APIs   │
└───────────────────────────────┘    └────────────────────────┘
```

### Roles de la nube:
- Almacenamiento desacoplado del cómputo (S3, ADLS, GCS).
- Compute escalable on-demand (Spark on Kubernetes, serverless SQL engines).
- Metadata y transacciones ACID en formatos abiertos (Delta / Iceberg / Hudi).

---
## 3. Patrones Arquitectónicos Relevantes

| Patrón | Descripción | Beneficio | Riesgo |
|--------|-------------|-----------|--------|
| Lambda | Doble pipeline: batch + stream | Flexibilidad histórica + tiempo real | Complejidad operativa |
| Kappa | Unifica procesamiento sobre log de eventos | Simplicidad conceptual | Reprocesos pueden ser costosos |
| Lakehouse | Fusiona Data Lake + Warehouse | SQL + ML en mismo storage | Gobernanza todavía madura |
| Medallion | Bronze/Silver/Gold por capas de refinamiento | Trazabilidad y calidad | Requiere disciplina |
| Data Mesh | Dominios + datos como producto | Escala organizacional | Coordinación de estándares |
| Streaming-first | Eventos como fuente central | Baja latencia y replay | Mayor complejidad inicial |

---
## 4. Almacenamiento Distribuido: Conceptos Clave

| Concepto | Explicación | Ejemplo |
|----------|------------|---------|
| Partición | División física por clave / rango temporal | year=2025/month=11/day=07 |
| Replicación | Copias redundantes para disponibilidad | 3 réplicas en HDFS / S3 multi-AZ |
| Formato Colunar | Organiza datos por columna, mejora compresión | Parquet / ORC |
| Metadata Layer | Administra transacciones y snapshots | Delta Log / Iceberg Manifest |
| Compaction | Reescritura de archivos pequeños | Optimiza lectura y costo |
| Z-Ordering | Organización multidimensional para pruning | Delta Lake ZORDER |

---
## 5. Cómputo Escalable y Paralelismo

El paralelismo divide un dataset en fragmentos procesados por workers independientes.

| Tipo | Descripción | Ejemplo | Beneficio |
|------|-------------|--------|----------|
| Paralelismo por Datos | Cada worker procesa una partición | Spark RDD/DataFrame Partition | Escala lineal |
| Paralelismo por Tarea | DAG de tareas independientes | Airflow paralelo | Utilización óptima |
| Paralelismo Vectorial | Operaciones SIMD / GPU | Rapids / Torch | Acelera ML / DL |
| Autoscaling | Ajuste dinámico de nodos | EMR / Dataproc autoscaling | Optimiza costo |

---
## 6. Simulador: Impacto de Particiones y Workers

Exploramos cómo cambia el tiempo estimado de procesamiento según particiones y número de workers.

In [ ]:
from dataclasses import dataclass
import math, random
from typing import List

@dataclass
class PartitionPlan:
    total_records: int
    partition_size: int
    workers: int
    scan_speed_records_per_sec: int = 250000  # velocidad promedio lectura por worker
    shuffle_factor: float = 0.15  # sobrecosto por agregaciones / joins
    skew_probability: float = 0.1  # probabilidad de partición desbalanceada

    def partitions(self) -> int:
        return math.ceil(self.total_records / self.partition_size)

    def _simulate_skew_multiplier(self) -> float:
        # Si hay skew una partición tarda más (entre 20% y 80%)
        if random.random() < self.skew_probability:
            return random.uniform(1.2, 1.8)
        return 1.0

    def estimated_time_seconds(self) -> float:
        base_partitions = self.partitions()
        # Particiones por worker
        partitions_per_worker = base_partitions / self.workers
        # Tiempo por partición (scan)
        scan_time_per_partition = self.partition_size / self.scan_speed_records_per_sec
        # Ajuste por skew (solo en la partición más lenta)
        skew = self._simulate_skew_multiplier()
        critical_path = partitions_per_worker * scan_time_per_partition * skew
        # Shuffle overhead global
        overhead = critical_path * self.shuffle_factor
        return critical_path + overhead

    def summary(self) -> str:
        return (f'Total Records: {self.total_records:,} | Partition Size: {self.partition_size:,} | ' 
                f'Partitions: {self.partitions()} | Workers: {self.workers} | ' 
                f'Estimated Time: {self.estimated_time_seconds():.2f}s')

def run_scenarios(total_records: int, partition_sizes: List[int], worker_counts: List[int]):
    print('='*70)
    print('Simulación de Planes de Particionamiento y Paralelismo')
    print('='*70)
    for ps in partition_sizes:
        for w in worker_counts:
            plan = PartitionPlan(total_records=total_records, partition_size=ps, workers=w)
            print(plan.summary())
    print('
Conclusión: más workers y particiones equilibradas reducen el tiempo, pero el skew puede afectar la ruta crítica.')

# Ejecutar simulación
run_scenarios(total_records=50_000_000, partition_sizes=[1_000_000, 5_000_000, 10_000_000], worker_counts=[4, 8, 16])

## 7. Mini Laboratorio: Elección de Formato

| Necesidad | Recomendado | Razón |
|-----------|-------------|-------|
| Consultas analíticas masivas | Parquet / ORC | Columna + compresión |
| Actualizaciones frecuentes | Delta / Iceberg / Hudi | ACID + schema evolution |
| ML sobre datos crudos | Parquet + DataFrame API | Vectorización |
| Integración streaming → lake | Hudi / Iceberg | Ingesta incremental + snapshots |
| Data sharing multi-cloud | Iceberg | Metadatos abiertos |
| Time travel / rollback | Delta / Iceberg | Snapshots y commits |

## 8. Checklist de Arquitectura Big Data Básica

- [ ] Estrategia clara de particionamiento temporal (año/mes/día)
- [ ] Formato colunar para capas Silver/Gold
- [ ] Compaction programado para archivos pequeños
- [ ] Métricas de ingestión: throughput, lag, error rate
- [ ] Monitor de skew y jobs lentos
- [ ] Catálogo y schema registry activos
- [ ] Estrategia de evolución de esquema documentada
- [ ] Backups y redundancia multi-AZ/multi-region

## 9. Preguntas de Reflexión

1. ¿Cuándo preferir Kappa sobre Lambda?
2. ¿Qué riesgos implica un diseño sin compaction?
3. ¿Cómo detectar skew en un job distribuido?
4. ¿Por qué separar Bronze/Silver/Gold ayuda a auditoría?

## 10. Próximos Pasos (Nivel Mid)

En el siguiente nivel compararemos el rendimiento de frameworks clásicos vs modernos (pandas vs polars/duckdb) y profundizaremos en Spark, Flink, Kafka, NoSQL y bases vectoriales.

---
**Referencias sugeridas (2025):**
- Apache Iceberg / Delta Lake / Hudi docs
- Whitepaper: Lakehouse Architecture
- Streaming Systems (Tyler Akidau)
- Data Mesh (Zhamak Dehghani)